<a href="https://colab.research.google.com/github/Estherbh/Classi-fy/blob/main/organiser_fermiers_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Organisation des fermiers --- noms et codes

Nettoie une liste brute de fermiers (nom + code d'identification) saisie à la main,
et produit un fichier Excel exploitable.

## ⚠️ Données personnelles

**Les données réelles ne sont pas dans ce notebook.** Elles sont lues depuis
`data/fermiers_brut.txt`, qui est exclu de git via `.gitignore`.

Ce fichier contient des noms de personnes réelles et leurs codes : il ne doit jamais
être commité ni publié. Sur Colab, téléverse-le manuellement dans la session.

## Format attendu

Une entrée par ligne. La saisie peut être irrégulière, le script tolère
(les codes ci-dessous sont fictifs) :

```
1.NOM PRENOM:PH-MGN-XX-0000
2 NOM PRENOM PH-MTG-XX 000
3. NOM PRENOM              <- nom sans code, accepté
```


In [ ]:
from pathlib import Path
import re

import pandas as pd

# Fichier source : hors git, contient des données personnelles.
FICHIER_SOURCE = Path("data/fermiers_brut.txt")
FICHIER_SORTIE = Path("data/liste_fermiers_avec_codes.xlsx")

if not FICHIER_SOURCE.exists():
    raise FileNotFoundError(
        f"{FICHIER_SOURCE} introuvable.\n"
        "Place la liste brute des fermiers dans ce fichier (une entrée par ligne).\n"
        "Sur Colab : téléverse-le dans la session, il ne doit pas être dans le dépôt."
    )

lignes = [
    ligne.strip()
    for ligne in FICHIER_SOURCE.read_text(encoding="utf-8").splitlines()
    if ligne.strip()
]

print(f"{len(lignes)} lignes lues depuis {FICHIER_SOURCE}")

In [ ]:
# Un code suit le schéma <PH|CO>-<MGN|MTG|MTH>-<zone>-<numéro>, par exemple
# PH-MGN-XX-0000 ou CO-MTG-XX 000 (exemples fictifs).
# La saisie réelle est irrégulière : espaces au milieu, préfixe PH parfois absent,
# tirets manquants. On accepte ces variantes, puis on normalise --- en gardant le code
# brut pour pouvoir vérifier ce que la normalisation a changé.
MOTIF_CODE = re.compile(r"((?:PH|CO)?\s*-?\s*(?:MGN|MTG|MTH)[\s\w-]*)$", re.IGNORECASE)


def nettoyer_nom(nom):
    """Retire la numérotation de début, la ponctuation de fin et les espaces multiples."""
    nom = re.sub(r"^\s*\d+\s*[.:)\-]*\s*", "", nom)
    nom = re.sub(r"[\s.:,\-]+$", "", nom)
    nom = re.sub(r"\s+", " ", nom).strip()
    return nom.title()


def normaliser_code(code):
    """Supprime les espaces internes : "PH-MTG- MU- N026" -> "PH-MTG-MU-N026"."""
    return re.sub(r"\s+", "", code).strip(" .-:").upper()


def analyser_ligne(ligne):
    """Retourne (nom, code_brut). code_brut vaut None si la ligne n'a pas de code."""
    correspondance = MOTIF_CODE.search(ligne)
    if not correspondance:
        return nettoyer_nom(ligne), None
    code_brut = correspondance.group(1).strip()
    nom = nettoyer_nom(ligne[: correspondance.start()])
    return nom, code_brut

In [ ]:
enregistrements = []
for ligne in lignes:
    nom, code_brut = analyser_ligne(ligne)
    if not nom and not code_brut:
        continue
    enregistrements.append(
        {
            "Nom du Fermier": nom,
            "Code": normaliser_code(code_brut) if code_brut else "",
            "Code brut": code_brut or "",
            "Ligne source": ligne,
        }
    )

df = pd.DataFrame(enregistrements)

print(f"Total des entrées : {len(df)}")
print(f"  avec code : {(df['Code'] != '').sum()}")
print(f"  sans code : {(df['Code'] == '').sum()}")
df.head(10)

In [ ]:
# Contrôles qualité.
# La version précédente de ce script ne les faisait pas : des doublons passaient
# silencieusement dans le fichier Excel final.

codes = df.loc[df["Code"] != "", "Code"]
codes_dupliques = codes[codes.duplicated(keep=False)]
if len(codes_dupliques):
    print(f"⚠️  {codes_dupliques.nunique()} code(s) attribué(s) à plusieurs fermiers :")
    colonnes = ["Nom du Fermier", "Code", "Ligne source"]
    print(df[df["Code"].isin(codes_dupliques)].sort_values("Code")[colonnes].to_string(index=False))
else:
    print("✅ Aucun code en double.")

noms = df.loc[df["Nom du Fermier"] != "", "Nom du Fermier"]
noms_dupliques = noms[noms.duplicated(keep=False)]
if len(noms_dupliques):
    print(f"\n⚠️  {noms_dupliques.nunique()} nom(s) apparaissant plusieurs fois :")
    for nom in sorted(noms_dupliques.unique()):
        print(f"   - {nom}")
else:
    print("\n✅ Aucun nom en double.")

sans_nom = df[df["Nom du Fermier"] == ""]
if len(sans_nom):
    print(f"\n⚠️  {len(sans_nom)} ligne(s) sans nom exploitable :")
    for ligne in sans_nom["Ligne source"]:
        print(f"   - {ligne}")

modifies = df[(df["Code"] != "") & (df["Code"] != df["Code brut"].str.upper())]
if len(modifies):
    print(f"\nℹ️  {len(modifies)} code(s) normalisé(s) (espaces/casse), à vérifier :")
    print(modifies[["Nom du Fermier", "Code brut", "Code"]].to_string(index=False))

In [ ]:
FICHIER_SORTIE.parent.mkdir(parents=True, exist_ok=True)
df.to_excel(FICHIER_SORTIE, index=False, engine="openpyxl")
print(f"Fichier créé : {FICHIER_SORTIE}  ({len(df)} fermiers)")

# Sur Colab uniquement : déclenche le téléchargement du fichier.
try:
    from google.colab import files

    files.download(str(FICHIER_SORTIE))
except ImportError:
    pass